# 04 コールドスタート — 新しい商品・新しいユーザーに、どう薦めるか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 評価がまだない新しい商品・ユーザーに、どう薦めるかを調べる回です。
> - **たとえ話**: 初めて来たお客さんには、まず「どのジャンルが好きですか」と聞いて薦める。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-7（コールドスタート）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約2分

## ゴール
- **コールドスタート**（評価が無い／少ない新しい商品・ユーザー）では、03 の行列分解も人気順も**役に立たない**ことを確かめる。
- 商品の特徴（**ジャンル**）を使う **コンテンツベース**と、行列分解を組み合わせた**ハイブリッド**を作る。
- 評価が**少しずつ集まる**と、各手法の価値がどう変わるかを見る。
- ジャンルの情報が**不正確**なとき、ハイブリッドが**害になる**ことがあるのを確かめる。

## 前のNotebookとのつながり
03 の ALS は、評価の点数を使って個人化が効きました。ただし、**学習時に評価があった**ユーザーと商品にしか使えません。
現実の推薦では、**毎日新しい商品とユーザーが現れる**ので、評価が無い相手にも薦める必要があります。

## キーワード
- **コールドスタート**: 評価が無い（少ない）新しい商品／ユーザーに薦める問題。
- **コンテンツベース**: 商品の**特徴**（ここではジャンル）と、ユーザーの好みを結び付けて薦める。評価が無くても使える。
- **ハイブリッド**: 評価（協調）と特徴（コンテンツ）を**組み合わせる**。
- **事前分布**: 評価が無いときの**初期値**。ここでは「同じジャンルの既存商品の平均」。
- **フォールドイン**: 学習済みの因子を固定して、新しい1人（1商品）ぶんだけを、少ない評価から解く。

## 実験の設計
- **新しい商品**: 商品の20%を「新しい商品」にし、学習から外す（評価0件）。候補は新しい商品だけ。評価は**本当の好み**での precision@10。
- **新しいユーザー**: ユーザーの20%を「新しいユーザー」にし、**学習期間の最初のn件だけ**を与える。
- 世界を変えて10回、平均を出す。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase4

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 200)

e1 = run_phase4.e1()
e1

**出力の見方**（新しい商品・評価0件）:
- ランダムは **0.244**（新しい商品のうち、本当に好きなものの割合）。
- **人気順・平均評価順・ALS は、すべて 0.243 で、ランダムと同じ**です。評価が無い商品は、どれも**同点**（または因子0）で、順位を付けられないためです。03 で最も良かった ALS も、ここでは**無力**です。
- **ジャンル好み**（ユーザーが各ジャンルに付けた評価の平均）は **0.486**で、ランダムの約2倍。評価0件でも、**ジャンルという特徴**だけで薦められます。
- **ALS＋ジャンル事前分布**（ハイブリッド）は **0.504**。ジャンルの平均因子を初期値にするので、ジャンル好みと同程度（差は誤差の範囲: 標準誤差が 0.017〜0.022）です。
- **上限（ジャンルだけで分かる最良）** は **0.631**。ジャンルが同じ商品は区別できないので、ジャンルだけでは**ここまで**が限界です。本当の好みを知っていれば **0.895**。

In [ ]:
e2 = run_phase4.e2()
e2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(e2))
for col, st in [("ALS（因子0から）", "o-"), ("ALS＋ジャンル事前分布", "s-"), ("ジャンル好みだけ", "^--")]:
    ax[0].plot(x, e2[col], st, label=col)
ax[0].set_xticks(x); ax[0].set_xticklabels(e2["新しい商品の評価数"])
ax[0].set_xlabel("新しい商品に集まった評価数"); ax[0].set_ylabel("本当のprecision@10"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("評価が集まると、各手法はどうなるか")
ax[1].errorbar(x, e2["ハイブリッド−ALS（差）"], yerr=2 * e2["差の標準誤差"], fmt="o-", capsize=4)
ax[1].set_xticks(x); ax[1].set_xticklabels(e2["新しい商品の評価数"])
ax[1].set_xlabel("新しい商品に集まった評価数"); ax[1].set_ylabel("ハイブリッド − ALS（誤差棒は±2標準誤差）"); ax[1].grid(alpha=0.3)
ax[1].set_title("ジャンル事前分布の価値は、評価が増えると薄れる")
plt.tight_layout(); plt.show()

**出力の見方**（評価が少しずつ集まると）:
- **ALS（因子0から）** は、評価0件で 0.243、**1件で 0.441**、5件で 0.572、50件で 0.660 と上がります。**1件の評価で急に良くなる**のは、商品の「質」（バイアス）が、1件でも分かるためです。
- **ハイブリッド**は、**どの評価数でもALSを上回ります**（差は 0.261 → 0.115 → 0.072 → 0.043 → 0.031 と縮む。標準誤差は 0.003〜0.016 なので、**すべて誤差ではない**）。
- ただし**差は評価が増えると縮みます**。評価が多ければ、**データ自体が語る**ので、事前分布の助けが要らなくなるためです。**事前分布が最も効くのは、評価が最も少ないとき**です。
- **ジャンル好みだけ**は、新しい商品の評価を使わないので、評価が増えても **0.48 で一定**で、頭打ちです。ALSだけ（因子0から）は、評価2件で 0.480（ジャンル好みと同程度）、**5件で 0.572 と上回ります**。
- 使い分け: **評価0件 → ジャンル（コンテンツ）／評価が少し → ハイブリッド／評価が十分 → 協調（ALS）中心**。

In [ ]:
e3 = run_phase4.e3()
e3

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(e3))
for col, st in [("人気順", "o-"), ("平均評価順", "s-"), ("ALS（フォールドイン）", "^-"), ("ジャンル好み", "d--")]:
    ax.plot(x, e3[col], st, label=col)
ax.set_xticks(x); ax.set_xticklabels(e3["新しいユーザーの評価数"])
ax.set_xlabel("新しいユーザーの評価数"); ax.set_ylabel("本当のprecision@10"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("新しいユーザー: 評価が増えるほど、個人化が効く")
plt.show()

**出力の見方**（新しいユーザー）:
- **評価0件**: 人気順 0.323、平均評価順 0.610、ALS（フォールドイン）0.635。ユーザーの情報が無いので、**全員に同じ商品を薦める**ことになります（平均評価順は、03 と同様に強い基準）。
- **評価が増えるほど、ALSが平均評価順を引き離す**: 差は 0.025（0件）→ 0.053（1件）→ 0.123（5件）→ 0.159（10件）→ 0.177（50件）。差は標準誤差の 4倍以上（0件でも 0.025 対 0.006）です。**5〜10件で、効果の大半**が得られます。
- **ジャンル好みだけ**は、評価が少ないと弱く（5件で 0.420）、50件で 0.488。**ALSには届きません**。
- 人気順は、評価が増えても**変わりません**（個人の情報を使わない）。むしろ少し下がります（0.323 → 0.302）。
- 実務の含意: 新しいユーザーには、**最初の数件の評価を集める工夫**（好みを尋ねる画面など）が、大きな価値を持つ。

In [ ]:
e4 = run_phase4.e4()
e4

**出力の見方**（新しい商品のジャンル札を、一部ランダムに付け替えたとき）:
- 札が正しい（付け替え0%）ときは、ハイブリッドがALSを上回る（0件で +0.261）。
- **付け替え割合が増えると、ハイブリッドの優位が縮み、逆転します**:
  - 評価0件: 札が1/2 間違っていても、まだ +0.130 で有利（そもそもALSは0.243=ランダムなので、間違った札でも、ほかに手がかりが無い）。
  - **評価5件**: 付け替え50%で **−0.007**（差は標準誤差の1倍。ほぼ同じ）、75%で **−0.045**、100%で **−0.088**（標準誤差は 0.010 → 明確に悪化）。
  - 評価20件: 100%で −0.039。
- 意味: **間違った事前分布は、データが語る内容を邪魔する**（正則化で、事前分布のほうへ引っ張られるため）。**特徴の質が悪い**なら、ハイブリッドは害になります。
- 同じ理由で、**評価が多いときほど害は小さい**（データが事前分布に打ち勝つ）。

## 試してみよう（演習）
1. **新しい商品の割合を変える**: `run_phase4.COLD_FRAC = 0.5` にして `e1()` を実行してください。分かるのは、既存商品が減ると、ジャンルの平均因子が不安定になり、ハイブリッドが弱くなるかどうかです。
2. **事前分布の強さを変える**: `fold_in_items(..., reg=5.0, reg_bias=10.0)` の値を、`reg=1.0` や `reg=20.0` にして `e2` と同じ実験をしてください。分かるのは、正則化が強いほど、少ない評価ではジャンルの平均に引っ張られ、多い評価では遅れることです。
3. **ジャンル好みの重み**: `genre_profile_scores(..., weight=3.0)` を `0.5` と `20.0` にして、E1のジャンル好みを比べてください。分かるのは、ユーザーの評価が少ないジャンルを、どれだけ全体平均に寄せるかの影響です。
4. **新しいユーザーの与え方**: E3で「最初のn件」の代わりに「ランダムなn件」を与えるには、`sort_values("time")` を `sample(frac=1)` に変えます。分かるのは、最初の評価が人気商品に偏る（露出の偏り）ことの影響です。

## まとめ
- **評価が無い**新しい商品には、人気順・平均評価順・ALSは**ランダムと同じ**（0.243）。**ジャンル（コンテンツ）** を使うと **0.486** に上がる（ジャンルだけの上限は 0.631）。
- **ハイブリッド**（ALS＋ジャンル事前分布）は、評価が少ないほど価値が大きく（評価0件 +0.261、5件 +0.072、50件 +0.031）、**評価が増えると価値が薄れる**。
- **新しいユーザー**では、評価が5〜10件集まると個人化の効果の大半が得られる（平均評価順との差 +0.12〜+0.16）。
- ジャンルの札が**不正確**だと、ハイブリッドは**害になる**（評価5件・札が全部ランダムだと −0.088）。**特徴の質**を疑う。

次の Phase5 では、**人気への偏り・露出の偏り**を補正する方法（逆確率重み付け）と、**推薦を繰り返すと偏りが増幅する**ことを調べます。